### __Import packages__

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.viterbi import *
import pandas as pd

### __Get testing data__

In [3]:
test_set = pd.read_pickle("../data/processed/test_data/test_set.pkl")

### __Initialize states and model__

In [4]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

### __Make test sequence (track 1039)__

In [5]:
sequence = test_set[test_set['TrackID'] == 1039]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None]

### __Run Viterbi model of first sequence__

In [6]:
path, probability = model.viterbi(AIS_sequence, RF_sequence)

In [7]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(30, 'RF'), (126, 'RF'), (195, 'RF'), (308, 'RF')],       
The indexes of RF and/or M of the real path are: [(30, 'RF'), (126, 'RF'), (195, 'RF'), (308, 'M')]


### __Make test sequence two (track 76)__

In [8]:
sequence2 = test_set[test_set['TrackID'] == 76]

AIS_sequence2 = [(time, coord) for time, coord in zip(sequence2["AIS_Timestamp"], sequence2["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence2 = [(time, coord) for time, coord in zip(sequence2["RF_Timestamp"], sequence2["RF"]) 
               if pd.notna(time) and coord is not None]

In [9]:
path2, probability2 = model.viterbi(AIS_sequence2, RF_sequence2)

In [10]:
RF_M_indexes_pred_path2 = [(i, state) for i, state in enumerate(path2) if state in {"RF", "M"}]
RF_M_indexes_real_path2 = [(i, state) for i, state in enumerate(sequence2["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path2}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path2}")

The indexes of RF and/or M of predicted path are: [(5, 'RF'), (150, 'RF')],       
The indexes of RF and/or M of the real path are: [(5, 'RF'), (150, 'RF')]


### __Run viterbi algorithm per RF observation__

In [ ]:
from haversine import haversine, Unit

In [155]:
RF_data = test_set[test_set["RF"].notna()].copy()
RF_data["RF_data_point_id"] = RF_data.groupby("TrackID").cumcount() + 1

viterbi_results = []
radius_km = 1

for _, RF_data_point in RF_data.iterrows():
    RF_coord = RF_data_point["RF"]
    RF_time = RF_data_point["Timestamp"]
    
    for AIS_id, AIS_data in test_set.groupby("TrackID"):
        AIS_coords = [coord for coord in AIS_data["AIS"] if coord is not None]
        RF_within_radius = any(haversine(RF_coord, coord) <= radius_km for coord in AIS_coords)
        
        min_AIS_time = AIS_data["Timestamp"].min()
        max_AIS_time = AIS_data["Timestamp"].max()
        RF_within_time = min_AIS_time <= RF_time <= max_AIS_time
        
        if RF_within_radius and RF_within_time:
            AIS_seq = [(time, coord) 
                       for time, coord in zip(AIS_data["Timestamp"], AIS_data["AIS"]) 
                       if pd.notna(time) and coord is not None]
            RF_seq = [(RF_time, RF_coord)]
            
            path, prob = model.viterbi(AIS_seq, RF_seq)
            
            viterbi_results.append(
                {
                    "RF_sequence_id": RF_data_point["TrackID"],
                    "RF_data_point_id": RF_data_point["RF_data_point_id"],
                    "AIS_sequence_id": AIS_id,
                    "viterbi_prob": prob,
                    "match": RF_data_point["TrackID"] == AIS_id,
                    "predicted_path": path
                })

In [164]:
df_results = pd.DataFrame(viterbi_results)
idx = df_results.groupby(["RF_sequence_id", "RF_data_point_id"])["viterbi_prob"].idxmax()
best_matches = df_results.loc[idx].reset_index(drop=True)
best_matches.head(5)

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,viterbi_prob,match,predicted_path
0,23,1,23,1.291772e-26,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
1,23,2,23,3.574493e-73,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2,23,3,23,2.930269e-75,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
3,29,1,29,8.986235e-32,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
4,29,2,29,2.817189e-32,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
